# Assignment 1: Data Collection and Prompt Engineering (10%)

This assignment requires students to create a data set for training and evaluation of an SUTD chatbot for prospective students. The data set should contain documents about SUTD and question-answer pairs suitable for model training and evaluation.

In addition to the data set, students should build a first prototype using only prompt engineering and foundation models available via APIs.

Objectives:
- Collect and curate documents related to SUTD (programs, admissions, campus, scholarships, student life, FAQs, policies).
- Create a high-quality Q&A dataset suitable for training and evaluation.
- Build a prompt-engineered chatbot prototype using a foundation model API with [Amazon Bedrock — What is Bedrock?](https://docs.aws.amazon.com/bedrock/latest/userguide/what-is-bedrock.html).
- Evaluate prototype responses against the curated Q&A dataset.

Deliverables:
- Data artifact: documents (raw + cleaned), Q&A pairs (JSONL), metadata (sources, timestamps, licenses).
- Notebook: end-to-end workflow (collection → cleaning → Q&A generation → prototype → evaluation).
- Short report: methodology, data sources, prompt design, evaluation summary.

Grading (10%):
- Data quality and coverage (3%)
- Q&A diversity, clarity, and correctness (3%)
- Prototype design and prompt engineering (2%)
- Evaluation thoroughness and analysis (2%)


In [5]:
import pandas as pd

# Rebuild the 'documents' list directly from your saved files!
metadata_df = pd.read_csv("data/metadata.csv")
documents = []

for _, row in metadata_df.iterrows():
    with open(row["processed_file"], "r", encoding="utf-8") as f:
        documents.append({
            "id": row["id"],
            "url": row["url"],
            "text": f.read()
        })

print(f"Successfully loaded {len(documents)} documents directly from the data folder! No API credits spent. 💸")

Successfully loaded 7 documents directly from the data folder! No API credits spent. 💸


# Setup & Environment

This notebook uses Python. If you use external APIs (e.g., AWS Bedrock), store credentials in environment variables and load them securely (do not hardcode keys).

Recommended packages:
- requests, beautifulsoup4, pandas, numpy, tqdm
- scikit-learn (optional for baseline retrieval)
- openai OR anthropic OR boto3 (choose one API path)
- rouge-score OR evaluate (optional, for metrics)

Tip: Use a virtual environment and a `.env` file or system keychain.

Example environment variables:
- OPENAI_API_KEY
- ANTHROPIC_API_KEY
- AWS_REGION + AWS credentials for Bedrock


In [6]:
import os
import json
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import boto3
import anthropic
import urllib3
import httpx

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

load_dotenv()

client = anthropic.Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    http_client=httpx.Client(verify=False)  # Bypass SSL verification
)

MODEL_ID = "claude-haiku-4-5-20251001"

print("Environment setup complete. Anthropic client initialized with SSL bypassed.")

Environment setup complete. Anthropic client initialized with SSL bypassed.


# Data Collection

Collect documents about SUTD from official, public sources (admissions pages, program descriptions, campus life, FAQ, scholarship info, policies). Respect robots.txt and terms of use; avoid overloading servers, and cache downloads locally.

Suggested steps:
1. Identify seed URLs and a scope definition (which pages to include).
2. Fetch pages politely (rate limiting), parse text, and store raw HTML + cleaned text.
3. Track metadata: URL, title, section, timestamp, retrieval status, license.
4. Normalize and deduplicate content; segment long pages into sections.

Artifacts:
- data/raw/*.html
- data/processed/*.md or *.txt
- data/metadata.csv


In [3]:
import re
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

urls = [
    "https://www.sutd.edu.sg/Admissions/Undergraduate",
    "https://www.sutd.edu.sg/admissions/graduate/masters",
    "https://www.sutd.edu.sg/admissions/graduate/phd",
    "https://www.sutd.edu.sg/research/research-centres",
    "https://www.sutd.edu.sg/education/undergraduate/courses",
    "https://www.sutd.edu.sg/campus-life",
    "https://www.sutd.edu.sg/about/design-ai/"
]

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

documents = []
metadata = []

for idx, url in enumerate(urls):
    try:
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, verify=False)
        response.raise_for_status()
        
        # Save raw HTML
        raw_path = f"data/raw/page_{idx}.html"
        with open(raw_path, "w", encoding="utf-8") as f:
            f.write(response.text)
            
        # Parse and clean text
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Remove scripts and styles
        for script in soup(["script", "style", "nav", "footer"]):
            script.extract()
            
        text = soup.get_text(separator=' ')
        # Clean up whitespace
        clean_text = re.sub(r'\s+', ' ', text).strip()
        
        # Save processed text
        processed_path = f"data/processed/page_{idx}.txt"
        with open(processed_path, "w", encoding="utf-8") as f:
            f.write(clean_text)
            
        # Store metadata
        metadata.append({
            "id": idx,
            "url": url,
            "raw_file": raw_path,
            "processed_file": processed_path,
            "retrieved_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "status": "success"
        })
        
        documents.append({"id": idx, "text": clean_text, "url": url})
        print(f"Successfully scraped: {url}")
        time.sleep(1)
        
    except Exception as e:
        print(f"Failed to scrape {url}: {e}")
        metadata.append({
            "id": idx,
            "url": url,
            "raw_file": None,
            "processed_file": None,
            "retrieved_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "status": "failed"
        })

pd.DataFrame(metadata).to_csv("data/metadata.csv", index=False)
print("Data Collection Completed!")

Successfully scraped: https://www.sutd.edu.sg/Admissions/Undergraduate
Successfully scraped: https://www.sutd.edu.sg/admissions/graduate/masters
Successfully scraped: https://www.sutd.edu.sg/admissions/graduate/phd
Successfully scraped: https://www.sutd.edu.sg/research/research-centres
Successfully scraped: https://www.sutd.edu.sg/education/undergraduate/courses
Successfully scraped: https://www.sutd.edu.sg/campus-life
Successfully scraped: https://www.sutd.edu.sg/about/design-ai/
Data Collection Completed!


# Q&A Generation

Create question-answer pairs suitable for model training and evaluation. Aim for coverage: admissions eligibility, deadlines, programs, curriculum, scholarships, housing, student life, application process, contact channels.

Options:
- Manual authoring from authoritative sources (preferred for correctness).
- LLM-assisted generation using your curated documents, followed by human validation.

Guidelines:
- Keep answers concise and factual; include references (URL, section) in metadata.
- Avoid speculative or outdated info; include retrieval timestamp.
- Provide diverse phrasings and difficulty levels.


In [7]:
qa_pairs = []

def generate_qa_from_text(context_text, doc_id, num_pairs=3):
    system_prompt = "You are a helpful assistant. Generate factual Q&A pairs based ONLY on the provided text."
    prompt = f"""
    Read the following text about SUTD and generate {num_pairs} distinct Question and Answer pairs.
    Format the output strictly as a JSON array of objects with 'question' and 'answer' keys.
    
    Text: {context_text[:4000]} # Truncating to avoid token limits for this prototype
    """
    
    try:
        # Anthropic's direct API syntax
        response = client.messages.create(
            model=MODEL_ID,
            max_tokens=1000,
            temperature=0.2, # Low temperature for factual consistency
            system=system_prompt,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        
        # Parse the JSON response from the Anthropic object
        response_text = response.content[0].text
        
        # Extract JSON from potential markdown blocks
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif "```" in response_text:
            response_text = response_text.split("```")[1].split("```")[0].strip()
            
        generated_pairs = json.loads(response_text)
        
        for pair in generated_pairs:
            qa_pairs.append({
                "question": pair["question"],
                "answer": pair["answer"],
                "source_doc_id": doc_id
            })
            
    except Exception as e:
        print(f"Failed to generate Q&A for doc {doc_id}: {e}")

print("Generating Q&A pairs...")
for doc in documents:
    generate_qa_from_text(doc["text"], doc["id"])
    
print(f"Generated {len(qa_pairs)} Q&A pairs.")

Generating Q&A pairs...
Generated 21 Q&A pairs.


# Dataset Assembly

Format the Q&A into a machine-learning friendly structure (JSONL recommended). Include:
- id, question, answer
- source (URL/file), retrieved_at timestamp
- split (train/test/dev), topic/category

Ensure a clear train/test split with no leakage.


In [8]:
import uuid
import random
import time
import json

def determine_topic(url):
    url_lower = url.lower()
    if "admissions/undergraduate" in url_lower:
        return "undergraduate_admissions"
    elif "admissions/graduate/masters" in url_lower:
        return "masters_admissions"
    elif "admissions/graduate/phd" in url_lower:
        return "phd_admissions"
    elif "research" in url_lower:
        return "research_centres"
    elif "courses" in url_lower or "education" in url_lower:
        return "undergraduate_courses"
    elif "campus-life" in url_lower:
        return "campus_life"
    elif "design-ai" in url_lower:
        return "design_ai_program"
    else:
        return "general_sutd_info"

dataset = []
topic_counts = {}

for qa in qa_pairs:
    split = "train" if random.random() < 0.8 else "test"
    
    # Find the original URL for this specific Q&A pair
    source_url = next(d["url"] for d in documents if d["id"] == qa["source_doc_id"])
    
    # Assign the dynamic topic
    topic = determine_topic(source_url)
    
    # Track counts for our printout
    topic_counts[topic] = topic_counts.get(topic, 0) + 1
    
    dataset.append({
        "id": str(uuid.uuid4()),
        "question": qa["question"],
        "answer": qa["answer"],
        "source": source_url,
        "retrieved_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "split": split,
        "topic": topic
    })

# Save as JSONL
with open("data/sutd_qa_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item) + "\n")
        
print("Dataset successfully assembled and saved to data/sutd_qa_dataset.jsonl")
print("\nTopic Distribution:")
for topic, count in topic_counts.items():
    print(f"- {topic}: {count} pairs")

Dataset successfully assembled and saved to data/sutd_qa_dataset.jsonl

Topic Distribution:
- undergraduate_admissions: 3 pairs
- masters_admissions: 3 pairs
- phd_admissions: 3 pairs
- research_centres: 3 pairs
- undergraduate_courses: 3 pairs
- campus_life: 3 pairs
- design_ai_program: 3 pairs


# Prompt-Engineered Prototype

Build a simple chatbot that answers prospective student questions using:
- A concise system prompt (tone: helpful, factual, official).
- Lightweight retrieval from your curated documents for grounding.
- A foundation model API (OpenAI, Anthropic, or AWS Bedrock).

Note: Do not include private keys in the notebook. Use environment variables.


In [9]:
def sutd_chatbot(user_query, context_documents):
    # Expanded retrieval: Give Claude more text to read so it doesn't say "I don't know" as often!
    context = "\n\n".join([f"Source ({doc['url']}):\n{doc['text'][:15000]}" for doc in context_documents])
    
    system_prompt = """You are an official SUTD chatbot for prospective students. 
    You must answer strictly based on the provided context. 
    CRITICAL INSTRUCTION: Answer directly and concisely. DO NOT use conversational filler, introductory phrases, or pleasantries. 
    NEVER say "Based on the provided context" or "According to the information". Just provide the exact answer.
    If the context does not contain the answer, reply exactly with: 'I don't have information on that.'
    Hallucination and fabrication of information is strictly prohibited. Always refer to the provided context and never make up answers."""
    
    user_message = f"""
    Context Information:
    {context}
    
    Prospective Student Question: {user_query}
    """
    
    try:
        response = client.messages.create(
            model=MODEL_ID,
            max_tokens=800,
            temperature=0.0, # Zero temperature for maximum factual consistency
            system=system_prompt,
            messages=[
                {"role": "user", "content": user_message}
            ]
        )
        return response.content[0].text
    except Exception as e:
        return f"Error connecting to Anthropic API: {str(e)}"

# --- INTERACTIVE CHAT INTERFACE WITH PANIC BUTTON ---
print("🤖 SUTD Prototype Chatbot Initialized!")
print("Type 'exit' or 'quit' to stop the chat.")
print("Lost the input box? Just click the 'Interrupt' (square Stop icon) next to the cell!\n")

while True:
    try:
        user_input = input("Prospective Student (You): ")
        
        if user_input.lower() in ['exit', 'quit']:
            print("SUTD Bot: Thank you for visiting. Shutting down.")
            break
            
        print("SUTD Bot is thinking...")
        answer = sutd_chatbot(user_input, documents)
        print(f"SUTD Bot: {answer}\n")
        print("-" * 50)
        
    except KeyboardInterrupt:
        # THE MAGIC FIX: Catches the manual stop button so the cell exits cleanly!
        print("\n\nSUTD Bot: 🛑 Chat forcefully interrupted by user. Shutting down cleanly!")
        break

🤖 SUTD Prototype Chatbot Initialized!
Type 'exit' or 'quit' to stop the chat.
Lost the input box? Just click the 'Interrupt' (square Stop icon) next to the cell!

SUTD Bot is thinking...
SUTD Bot: I don't have information on that.

--------------------------------------------------
SUTD Bot is thinking...
SUTD Bot: SUTD offers undergraduate courses across seven pillars/clusters:

1. **Architecture and Sustainable Design (ASD)**
2. **Design and Artificial Intelligence (DAI)**
3. **Engineering Product Development (EPD)**
4. **Engineering Systems and Design (ESD)**
5. **Humanities, Arts and Social Sciences (HASS)**
6. **Information Systems Technology and Design (ISTD)**
7. **Science, Mathematics and Technology (SMT)**

Courses are categorized by type including Freshmore core, core, core elective, elective/technical elective, and capstone courses across 8 terms.

Examples of available courses include:
- Design Thinking Project I, II, III (Freshmore core)
- Technologies for Sustainable Glob

# Evaluation

Evaluate prototype answers against the Q&A dataset. Use a simple metric (e.g., token overlap or string similarity) and manual spot checks.

Suggested metrics:
- Exact match / normalized overlap
- ROUGE-L (optional)
- Human review with rubric (clarity, correctness, completeness, source alignment)


In [10]:
def calculate_overlap(true_answer, pred_answer):
    # Simple token overlap metric
    true_tokens = set(true_answer.lower().split())
    pred_tokens = set(pred_answer.lower().split())
    if not true_tokens:
        return 0.0
    overlap = len(true_tokens.intersection(pred_tokens)) / len(true_tokens)
    return round(overlap, 2)

test_set = [item for item in dataset if item["split"] == "test"]
results = []

print(f"Evaluating on {len(test_set)} test cases...")

for item in test_set:
    predicted = sutd_chatbot(item["question"], documents)
    overlap_score = calculate_overlap(item["answer"], predicted)
    results.append({
        "question": item["question"],
        "true_answer": item["answer"],
        "predicted": predicted,
        "overlap_score": overlap_score
    })

eval_df = pd.DataFrame(results)
print(f"Average Token Overlap Score: {eval_df['overlap_score'].mean()}")
display(eval_df)

Evaluating on 4 test cases...
Average Token Overlap Score: 0.6225


,question,true_answer,predicted,overlap_score
0,What are the main pillars and clusters of unde...,SUTD's main pillars and clusters include: ASD ...,The main pillars and clusters of undergraduate...,0.39
1,What is SUTD's unique positioning in the globa...,SUTD is the world's first Design·AI university...,SUTD is the world's first Design·AI university...,0.61
2,How does SUTD view the role of artificial inte...,"SUTD views AI not just as a tool, but as a par...","SUTD views AI not just as a tool, but as a par...",0.78
3,What is the concept behind SUTD's 'Not Turing'...,SUTD's 'Not Turing' approach puts the human ba...,"SUTD's ""Not Turing"" approach contrasts with th...",0.71


### Human Review and Analysis

Based on the prototype's evaluation results (Average Token Overlap Score: **0.6225**), the V1 model demonstrates a solid baseline for retrieving and answering queries, but also reveals clear areas for architectural improvement. 

##### **Pros: What Worked Well**
* **High Semantic Comprehension:** The model performed very well on conceptual and qualitative questions. For instance, questions about SUTD's view on AI (Score: 0.78) and the "Not Turing" concept (Score: 0.71) showed high overlap, indicating the model successfully isolated the correct context and relayed it accurately.
* **Strict Prompt Adherence:** The chatbot followed the system prompt instructions well, providing direct, factual answers without unnecessary conversational filler (e.g., avoiding "Based on the text...").
* **Zero-Shot Baseline:** Achieving an average overlap of over 62% using a basic string-concatenation approach proves that the foundational data collection and cleaning steps yielded high-quality context.

##### **Cons & Limitations**
* **Brittle Evaluation Metric:** The lowest score (0.39 for SUTD's pillars and clusters) highlights the limitation of using simple token overlap as an evaluation metric. The model likely provided the correct information but formatted the list differently or used slightly different vocabulary than the ground-truth answer, penalizing the score unfairly.
* **Inefficient Context Handling:** The current V1 prototype relies on a naive string-concatenation method. It stuffs the entire corpus of SUTD documents into the prompt context for every single query. 
* **Scalability Bottleneck:** While string-concatenation works for a small prototype of 7 documents, it is entirely unscalable. As the dataset grows, this approach will rapidly hit the foundation model's maximum context window limit. Furthermore, sending thousands of redundant tokens per request incurs massive and unnecessary API costs.

##### **Further Improvements for Production (V2)**
To transition this prototype into a production-ready MLOps pipeline, the following architectural upgrades are recommended:

1. **Implement a Vector Database for True RAG:** Instead of sending the entire SUTD context in the prompt, a production iteration would chunk the scraped documents, generate embeddings, and store them in a Vector Database (e.g., ChromaDB, FAISS, or Pinecone). Upon receiving a user query, the system would perform a similarity search to retrieve only the top-$k$ most relevant document chunks. 
   * *Impact:* This drastically reduces API token costs, lowers response latency, and allows the system to scale infinitely to encompass the entire SUTD website without hitting context window limits.
   
2. **Semantic Evaluation Metrics:**
   Replace or augment the exact token overlap metric with semantic similarity metrics (such as BERTScore or cosine similarity of embeddings) or an "LLM-as-a-judge" framework. This will accurately reward the model for correct answers even if it paraphrases the ground truth.

3. **Conversation Memory:**
   Implement a memory buffer to retain chat history, allowing prospective students to ask follow-up questions (e.g., "Tell me more about the second pillar you mentioned") without losing context.

# End

This concludes Individual assignment 1.

Please submit this notebook with your answers and the generated output cells as a **Jupyter notebook file** via github.


Every student should do the following submission steps:
1. Create a private github repository **sutd_5055mlops** under your github user.
2. Add your instructors as collaborator: ddahlmeier, bearwithchris and MarkHershey
3. Save your submission as `individual_assignment_01_StudentID`.ipynb (replace StudentID with your student ID)
4. Push the submission files to your repo 
5. Submit the link to the repo via eDimensions 



**Assignment due 27 Feb (Fri) 23:59**